In [20]:
import pandas as pd
import geopandas as gpd
from neo4j import GraphDatabase
from shapely.geometry import Point, Polygon
import h3
import folium

gdf = gpd.read_file("lombardia.geojson")

# Sostituisci questi segnaposto con le tue credenziali Neo4j Aura
# Il tuo URI di Aura sarà simile a neo4j+s://xxxxxxx.databases.neo4j.io
NEO4J_URI      = "neo4j+s://e50dcde9.databases.neo4j.io"
NEO4J_USER     = "e50dcde9"
NEO4J_PASSWORD = "5Wq7pNsDXym3-uMbo0Ug0BwK_dkTmko-B0NdiDJAg0k"
NEO4J_DATABASE = "e50dcde9" # O il nome del tuo database se diverso

# Ora possiamo tentare la connessione
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    print(f"Connesso a {NEO4J_URI}")
except Exception as e:
    print(f"Errore di connessione a Neo4j: {e}")

def run(cypher, **params):
    """Cypher → pandas DataFrame."""
    with driver.session(database=NEO4J_DATABASE) as s:
        return pd.DataFrame([r.data() for r in s.run(cypher, **params)])

def run_write(cypher, **params):
    """Run a write query and return its summary counters."""
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.execute_write(
            lambda tx: tx.run(cypher, **params).consume().counters
        )

Connesso a neo4j+s://e50dcde9.databases.neo4j.io


In [21]:
point = gdf.sample_points(1)

lon = point.x[0]
lat = point.y[0]

print(lat)
print(lon)

45.97606429791364
10.327680073848196


In [22]:
query = '''
    MATCH (o:Ospedale)-[:HA_REPARTO]->(rep:Reparto)

    WHERE rep.Nome = "OSTETRICIA"

    WITH point({
        latitude: $lat,
        longitude: $lon
    }) AS p, o

    WITH point.distance(
        p,
        o.Coordinate
    )/1000 AS km, o

    RETURN round(km,2) AS km, o.Coordinate AS coord

    ORDER BY km
    LIMIT 5'''

output = run(query, lat=lat, lon=lon) 

output.head()

,km,coord
0,32.73,"(9.92275865, 45.89167965)"
1,39.88,"(10.32968894, 46.33431031)"
2,40.84,"(9.879429303, 46.1707552)"
3,40.99,"(10.0510524, 45.66237981)"
4,44.11,"(10.43793264, 45.58732834)"


In [23]:
m = folium.Map(
    location=[45.5,9.5],
    zoom_start=8
)

folium.GeoJson("lombardia.geojson").add_to(m)

folium.Marker(location=[lat, lon],
              icon=folium.Icon(
        color='blue',
        icon='user',
        prefix='fa'   # Font Awesome
    )).add_to(m)

for _, row in output.iterrows():
    coord = row['coord']
    folium.Marker(location=[coord.latitude, coord.longitude],
                  icon=folium.Icon(
        color='red',
        icon='hospital',
        prefix='fa'   # Font Awesome
    )).add_to(m)

m.save(
    "prova.html"
)